In [1]:
# ============================================================
# Notebook    : 13_statistical_significance.ipynb
# Description : Paired bootstrap significance testing for the
#               5 core comparisons underlying §4.4's claims:
#                 Case A: ① vs ③c   (LightGBM,   static vs +Agg)
#                 Case A: ③ vs ④b   (Transformer, static vs +Agg)
#                 Case B: B1 vs B2  (demographic vs +behavioral)
#                 Case C: C1 vs C2  (static vs +BonusMalus)
#
#               CRITICAL PREREQUISITE: paired bootstrap requires
#               both models' predictions to refer to the EXACT
#               SAME test rows in the EXACT SAME order. This is
#               NOT assumed — it is verified below, because:
#                 - LightGBM notebooks filter by IDpol membership
#                   (pandas default row order from source CSV)
#                 - Transformer notebooks build sequences from
#                   list(test_idpols), where test_idpols is a
#                   Python set with NO guaranteed iteration order
#               If saved prediction files can't be verified
#               row-aligned, this notebook REBUILDS the aligned
#               pair from source data rather than trusting the
#               saved .npz order.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm torch scikit-learn pandas numpy tqdm


# ============================================================
# 1. Common imports
# ============================================================
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

np.random.seed(42)
N_BOOTSTRAP = 1000
ALPHA = 0.05  # 95% CI


# ============================================================
# 2. Bootstrap utility — PAIRED, computes the distribution of
#    (AUC_model2 - AUC_model1) directly, not two separate CIs.
#    Same resampled row indices applied to BOTH models each
#    iteration, which is what makes this "paired" and valid for
#    correlated predictions on the same test set.
# ============================================================
def paired_bootstrap_auc_diff(y_true, prob_1, prob_2, n_boot=N_BOOTSTRAP, seed=42):
    """
    Returns: array of shape (n_boot,) — AUC(model2) - AUC(model1)
    for each bootstrap resample, plus the point estimate.
    """
    y_true = np.asarray(y_true)
    prob_1 = np.asarray(prob_1)
    prob_2 = np.asarray(prob_2)
    n = len(y_true)

    assert len(prob_1) == n and len(prob_2) == n, \
        f"Length mismatch: y_true={n}, prob_1={len(prob_1)}, prob_2={len(prob_2)}"

    point_auc_1 = roc_auc_score(y_true, prob_1)
    point_auc_2 = roc_auc_score(y_true, prob_2)
    point_diff = point_auc_2 - point_auc_1

    rng = np.random.RandomState(seed)
    diffs = np.zeros(n_boot)

    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)  # same resampled indices for both models
        yt_b = y_true[idx]

        # skip degenerate resamples with only one class present
        if len(np.unique(yt_b)) < 2:
            diffs[b] = np.nan
            continue

        auc_1_b = roc_auc_score(yt_b, prob_1[idx])
        auc_2_b = roc_auc_score(yt_b, prob_2[idx])
        diffs[b] = auc_2_b - auc_1_b

    diffs = diffs[~np.isnan(diffs)]
    ci_low, ci_high = np.percentile(diffs, [100*ALPHA/2, 100*(1-ALPHA/2)])
    p_value_approx = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())  # two-sided

    return {
        "point_auc_1": point_auc_1,
        "point_auc_2": point_auc_2,
        "point_diff": point_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "significant": not (ci_low <= 0 <= ci_high),  # CI excludes 0
        "p_value_approx": p_value_approx,
        "n_valid_boot": len(diffs),
    }


def print_result(label, result):
    sig_marker = "***" if result["significant"] else "n.s."
    print(f"\n[{label}]")
    print(f"  AUC 1 (baseline) : {result['point_auc_1']:.4f}")
    print(f"  AUC 2 (comparison): {result['point_auc_2']:.4f}")
    print(f"  Diff             : {result['point_diff']:+.4f}")
    print(f"  95% CI           : [{result['ci_low']:+.4f}, {result['ci_high']:+.4f}]")
    print(f"  Significant (CI excludes 0): {sig_marker}")
    print(f"  Approx p-value   : {result['p_value_approx']:.4f}")
    print(f"  Valid bootstrap iterations: {result['n_valid_boot']}/{N_BOOTSTRAP}")


# ============================================================
# 3. CASE A — rebuild ①③c and ③④b aligned pairs from SOURCE
#    DATA (not saved .npz), to guarantee row alignment rather
#    than assume it.
# ============================================================
print("="*60)
print("CASE A — rebuilding aligned test sets from source")
print("="*60)

df_a_static = pd.read_csv("data/fremotor_multi_history_features.csv")
df_a_agg    = pd.read_csv("data/fremotor_multi_history_aggregate.csv")

with open("data/sequences/split_idpols.json", "r", encoding="utf-8") as f:
    split_ids = json.load(f)
test_idpols_a = split_ids["test"]  # keep as LIST (preserves order), not set

# ------------------------------------------------------------
# 3a. LightGBM pair: ① (static) vs ③c (aggregate)
#     Both filtered from their respective CSVs using the SAME
#     IDpol list, then sorted identically (IDpol, Year) so row
#     order is deterministic and shared — this is the alignment
#     guarantee, built explicitly rather than trusted.
# ------------------------------------------------------------
test_id_set_a = set(test_idpols_a)

df_a_static_test = df_a_static[df_a_static["IDpol"].isin(test_id_set_a)].sort_values(
    ["IDpol", "Year"]).reset_index(drop=True)
df_a_agg_test = df_a_agg[df_a_agg["IDpol"].isin(test_id_set_a)].sort_values(
    ["IDpol", "Year"]).reset_index(drop=True)

# VERIFY alignment before trusting it
alignment_ok = (
    len(df_a_static_test) == len(df_a_agg_test) and
    (df_a_static_test["IDpol"].values == df_a_agg_test["IDpol"].values).all() and
    (df_a_static_test["Year"].values == df_a_agg_test["Year"].values).all() and
    (df_a_static_test["Label"].values == df_a_agg_test["Label"].values).all()
)
print(f"\n[CHECK A-LGBM] Row alignment verified (IDpol+Year+Label match): {alignment_ok}")
assert alignment_ok, "STOP — LightGBM static/aggregate test rows do not align!"

for col in ["Usage", "VehType", "VehPower"]:
    df_a_static_test[col] = df_a_static_test[col].astype("category")
    df_a_agg_test[col] = df_a_agg_test[col].astype("category")

model_1 = lgb.Booster(model_file="data/sequences/lightgbm_static_model.txt")
model_3c = lgb.Booster(model_file="data/sequences/lightgbm_aggregate_model.txt")

X1 = df_a_static_test[["Expo", "Usage", "VehType", "VehPower"]]
X3c = df_a_agg_test[["Expo", "YearGap", "CumClaimCount", "ClaimRateSoFar",
                       "YearsSinceFirstClaimLog", "PrevLabel", "HasPriorClaim",
                       "Usage", "VehType", "VehPower"]]

prob_1 = model_1.predict(X1)
prob_3c = model_3c.predict(X3c)
y_a = df_a_static_test["Label"].values

print(f"[CHECK A-LGBM] n={len(y_a)}, positive rate={y_a.mean()*100:.2f}%")

result_A_lgbm = paired_bootstrap_auc_diff(y_a, prob_1, prob_3c)
print_result("Case A — LightGBM: ① Static vs ③c Aggregate", result_A_lgbm)


# ------------------------------------------------------------
# 3b. Transformer pair: ③ (static) vs ④b (aggregate)
#     Rebuilt from scratch with EXPLICIT sorted order — avoids
#     the set-iteration-order risk entirely by never relying on
#     set() for sequence construction here.
# ------------------------------------------------------------
print("\n[CHECK A-Transformer] Rebuilding sequences with guaranteed order...")

test_idpols_a_sorted = sorted(test_id_set_a)  # deterministic order, list not set

with open("data/sequences/vocabs.json", "r", encoding="utf-8") as f:
    vocabs = json.load(f)
vocab_sizes = {k: len(v) for k, v in vocabs.items()}
CAT_COLS = list(vocabs.keys())

DEVICE = torch.device("cpu")
torch.manual_seed(42)

class TimestepRiskTransformer(nn.Module):
    def __init__(self, vocab_sizes, n_numeric, emb_dim=64, n_heads=4,
                 n_layers=2, ff_dim=128, dropout=0.1):
        super().__init__()
        cat_cols = list(vocab_sizes.keys())
        self.cat_cols = cat_cols
        self.embeddings = nn.ModuleDict({
            col: nn.Embedding(vocab_sizes[col], emb_dim, padding_idx=0)
            for col in cat_cols
        })
        self.numeric_proj = nn.Linear(n_numeric, emb_dim)
        combined_dim = emb_dim * (len(cat_cols) + 1)
        self.input_proj = nn.Linear(combined_dim, emb_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(emb_dim // 2, 1),
        )

    def forward(self, numeric, cat_idx, attention_mask):
        cat_embeds = []
        for i, col in enumerate(self.cat_cols):
            cat_embeds.append(self.embeddings[col](cat_idx[:, :, i]))
        cat_embeds = torch.cat(cat_embeds, dim=-1)
        num_embed = self.numeric_proj(numeric)
        combined = torch.cat([cat_embeds, num_embed], dim=-1)
        x = self.input_proj(combined)
        T = x.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        padding_mask_float = torch.zeros_like(attention_mask, dtype=torch.float32)
        padding_mask_float.masked_fill_(~attention_mask, float("-inf"))
        encoded = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask_float)
        logits = self.classifier(encoded).squeeze(-1)
        return logits


def get_transformer_flat_predictions(model_path, df_source, numeric_cols, n_numeric, idpol_order):
    """
    Build sequences in the EXACT idpol_order given (a sorted list,
    not a set), run inference, flatten in that same order. Returns
    (y_flat, prob_flat) with row order fully determined by
    idpol_order + within-person Year order — no set() involved.
    """
    df_sorted = df_source.sort_values(["IDpol", "Year"])
    for col in CAT_COLS:
        df_sorted[col + "_idx"] = df_sorted[col].map(vocabs[col])
    grouped = {k: v for k, v in df_sorted.groupby("IDpol")}

    model = TimestepRiskTransformer(vocab_sizes=vocab_sizes, n_numeric=n_numeric).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    all_logits, all_labels = [], []
    CAT_IDX_COLS = [c + "_idx" for c in CAT_COLS]

    with torch.no_grad():
        for idpol in idpol_order:
            sub = grouped[idpol]
            numeric = torch.tensor(sub[numeric_cols].values.astype(np.float32)).unsqueeze(0)
            cat_idx = torch.tensor(sub[CAT_IDX_COLS].values.astype(np.int64)).unsqueeze(0)
            label = sub["Label"].values.astype(np.int64)
            mask = torch.ones(1, len(sub), dtype=torch.bool)

            logits = model(numeric, cat_idx, mask).squeeze(0).numpy()
            all_logits.append(logits)
            all_labels.append(label)

    y_flat = np.concatenate(all_labels)
    logits_flat = np.concatenate(all_logits)
    prob_flat = 1 / (1 + np.exp(-logits_flat))
    return y_flat, prob_flat


y_a3, prob_3 = get_transformer_flat_predictions(
    "data/sequences/transformer_static_model.pt", df_a_static,
    ["Expo"], 1, test_idpols_a_sorted
)
y_a4b, prob_4b = get_transformer_flat_predictions(
    "data/sequences/transformer_aggregate_model.pt", df_a_agg,
    ["Expo", "YearGap", "CumClaimCount", "ClaimRateSoFar",
     "YearsSinceFirstClaimLog", "PrevLabel", "HasPriorClaim"], 7, test_idpols_a_sorted
)

alignment_ok_transformer = (
    len(y_a3) == len(y_a4b) and (y_a3 == y_a4b).all()
)
print(f"[CHECK A-Transformer] Row alignment verified (same idpol_order, same labels): "
      f"{alignment_ok_transformer}")
assert alignment_ok_transformer, "STOP — Transformer static/aggregate test rows do not align!"

result_A_transformer = paired_bootstrap_auc_diff(y_a3, prob_3, prob_4b)
print_result("Case A — Transformer: ③ Static vs ④b Aggregate", result_A_transformer)


# ============================================================
# 4. CASE B — rebuild B1/B2 aligned pair
# ============================================================
print("\n" + "="*60)
print("CASE B — rebuilding aligned test set from source")
print("="*60)

from sklearn.model_selection import train_test_split

df_b = pd.read_csv("data/car_insurance_preprocessed.csv")

NUMERIC_COLS_B1 = ["CREDIT_SCORE", "ANNUAL_MILEAGE", "CHILDREN", "MARRIED", "VEHICLE_OWNERSHIP"]
CAT_COLS_B = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
              "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"]
FLAG_COLS_B = ["CREDIT_SCORE_was_missing", "ANNUAL_MILEAGE_was_missing"]
BEHAV_COLS_B = ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]

FEATURE_COLS_B1 = NUMERIC_COLS_B1 + CAT_COLS_B + FLAG_COLS_B
FEATURE_COLS_B2 = FEATURE_COLS_B1 + BEHAV_COLS_B

X_b_full = df_b[FEATURE_COLS_B2].copy()  # superset, split once, subset per model
y_b_full = df_b["OUTCOME"].copy()

X_train_b, X_temp_b, y_train_b, y_temp_b = train_test_split(
    X_b_full, y_b_full, test_size=0.30, random_state=42, stratify=y_b_full
)
X_val_b, X_test_b, y_val_b, y_test_b = train_test_split(
    X_temp_b, y_temp_b, test_size=0.50, random_state=42, stratify=y_temp_b
)

for col in CAT_COLS_B:
    X_test_b[col] = X_test_b[col].astype("category")

print(f"[CHECK B] Single split, n={len(X_test_b)} — B1/B2 use the SAME rows by construction")

model_B1 = lgb.Booster(model_file="data/sequences/case_b_lightgbm_demographic_model.txt")
model_B2 = lgb.Booster(model_file="data/sequences/case_b_lightgbm_full_model.txt")

prob_B1 = model_B1.predict(X_test_b[FEATURE_COLS_B1])
prob_B2 = model_B2.predict(X_test_b[FEATURE_COLS_B2])
y_b = y_test_b.values

result_B = paired_bootstrap_auc_diff(y_b, prob_B1, prob_B2)
print_result("Case B: B1 Demographic vs B2 Full", result_B)


# ============================================================
# 5. CASE C — rebuild C1/C2 aligned pair
# ============================================================
print("\n" + "="*60)
print("CASE C — rebuilding aligned test set from source")
print("="*60)

df_c = pd.read_csv("data/fremtpl2_preprocessed.csv")

NUMERIC_COLS_C1 = ["Exposure", "VehPower", "VehAge", "DrivAge", "Density"]
CAT_COLS_C = ["VehBrand", "VehGas", "Area", "Region"]
FEATURE_COLS_C1 = NUMERIC_COLS_C1 + CAT_COLS_C
FEATURE_COLS_C2 = ["Exposure", "VehPower", "VehAge", "DrivAge", "Density", "BonusMalus"] + CAT_COLS_C

X_c_full = df_c[FEATURE_COLS_C2].copy()
y_c_full = df_c["Label"].copy()

X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_c_full, y_c_full, test_size=0.30, random_state=42, stratify=y_c_full
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c, y_temp_c, test_size=0.50, random_state=42, stratify=y_temp_c
)

for col in CAT_COLS_C:
    X_test_c[col] = X_test_c[col].astype("category")

print(f"[CHECK C] Single split, n={len(X_test_c)} — C1/C2 use the SAME rows by construction")

model_C1 = lgb.Booster(model_file="data/sequences/case_c_lightgbm_static_model.txt")
model_C2 = lgb.Booster(model_file="data/sequences/case_c_lightgbm_bonusmalus_model.txt")

prob_C1 = model_C1.predict(X_test_c[FEATURE_COLS_C1])
prob_C2 = model_C2.predict(X_test_c[FEATURE_COLS_C2])
y_c = y_test_c.values

result_C = paired_bootstrap_auc_diff(y_c, prob_C1, prob_C2)
print_result("Case C: C1 Static vs C2 BonusMalus", result_C)


# ============================================================
# 6. Consolidated results table — this becomes the annotated
#    version of Table1/Table2 for the paper (§4.4 with * for
#    significance)
# ============================================================
print("\n" + "="*60)
print("CONSOLIDATED SIGNIFICANCE TABLE")
print("="*60)

sig_table = pd.DataFrame([
    {"Comparison": "Case A (LightGBM): ① Static vs ③c Aggregate",
     "AUC_1": result_A_lgbm["point_auc_1"], "AUC_2": result_A_lgbm["point_auc_2"],
     "Diff": result_A_lgbm["point_diff"],
     "CI_low": result_A_lgbm["ci_low"], "CI_high": result_A_lgbm["ci_high"],
     "Significant": result_A_lgbm["significant"], "p_approx": result_A_lgbm["p_value_approx"]},
    {"Comparison": "Case A (Transformer): ③ Static vs ④b Aggregate",
     "AUC_1": result_A_transformer["point_auc_1"], "AUC_2": result_A_transformer["point_auc_2"],
     "Diff": result_A_transformer["point_diff"],
     "CI_low": result_A_transformer["ci_low"], "CI_high": result_A_transformer["ci_high"],
     "Significant": result_A_transformer["significant"], "p_approx": result_A_transformer["p_value_approx"]},
    {"Comparison": "Case B: B1 Demographic vs B2 Full",
     "AUC_1": result_B["point_auc_1"], "AUC_2": result_B["point_auc_2"],
     "Diff": result_B["point_diff"],
     "CI_low": result_B["ci_low"], "CI_high": result_B["ci_high"],
     "Significant": result_B["significant"], "p_approx": result_B["p_value_approx"]},
    {"Comparison": "Case C: C1 Static vs C2 BonusMalus",
     "AUC_1": result_C["point_auc_1"], "AUC_2": result_C["point_auc_2"],
     "Diff": result_C["point_diff"],
     "CI_low": result_C["ci_low"], "CI_high": result_C["ci_high"],
     "Significant": result_C["significant"], "p_approx": result_C["p_value_approx"]},
])

print(sig_table.to_string(index=False))

sig_table.to_csv("paper_assets/tables/Table3_Statistical_Significance.csv", index=False)
print("\nSaved: paper_assets/tables/Table3_Statistical_Significance.csv")


# ============================================================
# 7. Summary
# ============================================================
print("\n===== Statistical Significance Summary =====")
n_sig = sig_table["Significant"].sum()
print(f"Comparisons tested    : {len(sig_table)}")
print(f"Statistically significant (95% CI excludes 0): {n_sig}/{len(sig_table)}")
for _, row in sig_table.iterrows():
    marker = "SIGNIFICANT" if row["Significant"] else "NOT significant"
    print(f"  {row['Comparison']}: {row['Diff']:+.4f} [{row['CI_low']:+.4f}, {row['CI_high']:+.4f}] — {marker}")
print("===============================================")

CASE A — rebuilding aligned test sets from source

[CHECK A-LGBM] Row alignment verified (IDpol+Year+Label match): True
[CHECK A-LGBM] n=54755, positive rate=12.79%

[Case A — LightGBM: ① Static vs ③c Aggregate]
  AUC 1 (baseline) : 0.7686
  AUC 2 (comparison): 0.7919
  Diff             : +0.0233
  95% CI           : [+0.0204, +0.0263]
  Significant (CI excludes 0): ***
  Approx p-value   : 0.0000
  Valid bootstrap iterations: 1000/1000

[CHECK A-Transformer] Rebuilding sequences with guaranteed order...
[CHECK A-Transformer] Row alignment verified (same idpol_order, same labels): True

[Case A — Transformer: ③ Static vs ④b Aggregate]
  AUC 1 (baseline) : 0.7664
  AUC 2 (comparison): 0.7914
  Diff             : +0.0249
  95% CI           : [+0.0216, +0.0282]
  Significant (CI excludes 0): ***
  Approx p-value   : 0.0000
  Valid bootstrap iterations: 1000/1000

CASE B — rebuilding aligned test set from source
[CHECK B] Single split, n=1500 — B1/B2 use the SAME rows by construction

[Cas